# Reproduce Results
Loads saved checkpoints and JSON result files, then regenerates all figures and metrics
reported in the paper. **No retraining required.**

Run cells top-to-bottom. Figures are saved to `./figures/`.

In [ ]:
import json, os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

sys.path.insert(0, '.')   # ensure project root is on path

from utils.data import load_cmapss, FEATURE_COLS
from utils.metrics import rmse, phm_score, evaluate
from models.lstm_baseline import LSTMBaseline
from models.tft import TemporalFusionTransformer

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = './data/CMAPSSData'
CKPT_DIR = './checkpoints'
RES_DIR  = './results'
FIG_DIR  = './figures'
os.makedirs(FIG_DIR, exist_ok=True)

SUBSETS = ['FD001', 'FD003']
MODELS  = ['lstm', 'tft']
SENSOR_NAMES = FEATURE_COLS

print(f'Device: {DEVICE}')

## 1. Load Results JSON Files

In [ ]:
results = {}
for model in MODELS:
    for subset in SUBSETS:
        key = f'{model}_{subset}'
        path = os.path.join(RES_DIR, f'{key}.json')
        with open(path) as f:
            results[key] = json.load(f)
        print(f'{key}: RMSE={results[key]["test_rmse"]:.4f}  PHM={results[key]["test_phm"]:.2f}')

## 2. Results Table (Table 1 in report)

In [ ]:
print('\n' + '='*60)
print(f'{"Model":<10} {"Subset":<8} {"RMSE":>10} {"PHM Score":>12}')
print('-'*60)
for subset in SUBSETS:
    for model in MODELS:
        r = results[f'{model}_{subset}']
        print(f'{model.upper():<10} {subset:<8} {r["test_rmse"]:>10.4f} {r["test_phm"]:>12.2f}')
    print('-'*60)
print('='*60)

## 3. Training Curves (Figure 1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
colors = {'lstm': '#2196F3', 'tft': '#FF5722'}
styles = {'FD001': '-', 'FD003': '--'}

for ax, subset in zip(axes, SUBSETS):
    for model in MODELS:
        r = results[f'{model}_{subset}']
        epochs = [h['epoch'] for h in r['history']]
        val_rmse = [h['val_rmse'] for h in r['history']]
        ax.plot(epochs, val_rmse,
                color=colors[model], label=model.upper(),
                linewidth=2, alpha=0.85)
    ax.set_title(f'Validation RMSE — {subset}', fontsize=13)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('RMSE (cycles)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/training_curves.pdf', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.pdf')

## 4. Predicted vs. True RUL Scatter (Figure 2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
lim = 130

for col, subset in enumerate(SUBSETS):
    for row, model in enumerate(MODELS):
        ax = axes[row][col]
        r = results[f'{model}_{subset}']
        y_true = np.array(r['y_true'])
        y_pred = np.array(r['y_pred'])
        ax.scatter(y_true, y_pred, alpha=0.5, s=18,
                   color=colors[model], edgecolors='none')
        ax.plot([0, lim], [0, lim], 'k--', lw=1.2, label='Perfect')
        ax.set_xlim(0, lim); ax.set_ylim(0, lim)
        ax.set_xlabel('True RUL (cycles)')
        ax.set_ylabel('Predicted RUL (cycles)')
        ax.set_title(f'{model.upper()} / {subset}\nRMSE={r["test_rmse"]:.2f}', fontsize=11)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/pred_vs_true.pdf', dpi=150, bbox_inches='tight')
plt.show()
print('Saved pred_vs_true.pdf')

## 5. Cross-Subset Performance Gap (Figure 3 — Contribution 2)

In [ ]:
# Bar chart: FD001 vs FD003 RMSE for each model
x = np.arange(len(MODELS))
width = 0.3

fig, ax = plt.subplots(figsize=(7, 4))
for i, subset in enumerate(SUBSETS):
    rmse_vals = [results[f'{m}_{subset}']["test_rmse"] for m in MODELS]
    bars = ax.bar(x + i * width, rmse_vals, width,
                  label=subset, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x + width / 2)
ax.set_xticklabels([m.upper() for m in MODELS])
ax.set_ylabel('Test RMSE (cycles)')
ax.set_title('Cross-Subset Generalisation: FD001 vs FD003')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/cross_subset.pdf', dpi=150, bbox_inches='tight')
plt.show()
print('Saved cross_subset.pdf')

## 6. TFT Variable Selection Weights (Interpretability — Figure 4)

In [ ]:
# Load TFT checkpoint and compute mean variable importance over the test set
from utils.data import FEATURE_COLS

for subset in SUBSETS:
    ckpt_path = os.path.join(CKPT_DIR, f'tft_{subset}_best.pt')
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    _, _, test_loader, n_features = load_cmapss(
        DATA_DIR, subset=subset, seq_len=30, batch_size=64)

    model = TemporalFusionTransformer(n_features=n_features)
    model.load_state_dict(ckpt['model_state'])
    model = model.to(DEVICE).eval()

    all_weights = []
    with torch.no_grad():
        for x, _ in test_loader:
            x = x.to(DEVICE)
            w = model.get_variable_weights(x)  # (B, T, F)
            all_weights.append(w.mean(dim=1).cpu().numpy())  # avg over time → (B, F)

    mean_importance = np.concatenate(all_weights, axis=0).mean(axis=0)  # (F,)

    fig, ax = plt.subplots(figsize=(10, 4))
    idx = np.argsort(mean_importance)[::-1]
    ax.bar(range(len(FEATURE_COLS)), mean_importance[idx],
           color='#FF5722', alpha=0.8)
    ax.set_xticks(range(len(FEATURE_COLS)))
    ax.set_xticklabels([FEATURE_COLS[i] for i in idx], rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Mean Selection Weight')
    ax.set_title(f'TFT Variable Importance — {subset}')
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/variable_importance_{subset}.pdf', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved variable_importance_{subset}.pdf')

## 7. Final Summary

In [ ]:
print('\n=== All figures generated ===')
for f in sorted(os.listdir(FIG_DIR)):
    print(f'  ./figures/{f}')

print('\n=== Final Results Summary ===')
for subset in SUBSETS:
    lstm_rmse = results[f'lstm_{subset}']['test_rmse']
    tft_rmse  = results[f'tft_{subset}']['test_rmse']
    delta = lstm_rmse - tft_rmse
    pct   = 100 * delta / lstm_rmse
    winner = 'TFT' if delta > 0 else 'LSTM'
    print(f'  {subset}: LSTM={lstm_rmse:.4f}  TFT={tft_rmse:.4f}  '
          f'Δ={abs(delta):.4f} ({abs(pct):.1f}% better for {winner})')